# STAGE 1 Training - Kaggle T4
Target: mAP 80% → 82-86%

In [ ]:
%%bash
# Clone repo và cd vào folder
git clone https://github.com/Khanhhh239/Model_XVLM_Training.git
cd Model_XVLM_Training/trainv4
pwd

In [ ]:
import os
os.chdir('/kaggle/working/Model_XVLM_Training/trainv4')
print(f"Current dir: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q albumentations
!pip install -q -e .

In [ ]:
# List available datasets để lấy exact names
import os
print("📦 Available datasets:")
for name in os.listdir('/kaggle/input'):
    print(f"  - {name}")

In [ ]:
from pathlib import Path
import shutil

# Create directories
Path("data/checkpoints").mkdir(parents=True, exist_ok=True)

print("🔍 Looking for datasets...\n")

# Find checkpoint dataset (có thể là ckpt-30k-hard hoặc ckpt_30k_hard)
ckpt_datasets = [d for d in os.listdir('/kaggle/input') if 'ckpt' in d.lower()]
if ckpt_datasets:
    ckpt_dir = f"/kaggle/input/{ckpt_datasets[0]}"
    print(f"✓ Found checkpoint dataset: {ckpt_datasets[0]}")
    
    # List files
    ckpt_files = os.listdir(ckpt_dir)
    print(f"  Files: {ckpt_files}")
    
    # Copy best.pth
    if 'best.pth' in ckpt_files:
        shutil.copy(f"{ckpt_dir}/best.pth", "data/checkpoints/best.pth")
        print(f"  ✓ Copied best.pth")
else:
    print("❌ No checkpoint dataset found!")

# Find bbox dataset
bbox_datasets = [d for d in os.listdir('/kaggle/input') if 'bbox' in d.lower()]
if bbox_datasets:
    bbox_dir = f"/kaggle/input/{bbox_datasets[0]}"
    print(f"\n✓ Found bbox dataset: {bbox_datasets[0]}")
    
    bbox_files = os.listdir(bbox_dir)
    print(f"  Files: {bbox_files}")
    
    # Copy boxes (có thể .json hoặc .jsonl)
    boxes_file = [f for f in bbox_files if 'boxes' in f.lower()][0]
    shutil.copy(f"{bbox_dir}/{boxes_file}", f"data/{boxes_file}")
    print(f"  ✓ Copied {boxes_file}")
else:
    print("\n⚠️  No bbox dataset found!")

# Find aicity data
aicity_datasets = [d for d in os.listdir('/kaggle/input') if 'aicity' in d.lower() or '30k' in d.lower()]
if aicity_datasets:
    aicity_dir = f"/kaggle/input/{aicity_datasets[0]}"
    print(f"\n✓ Found data dataset: {aicity_datasets[0]}")
    
    aicity_files = os.listdir(aicity_dir)
    print(f"  Files: {aicity_files}")
    
    # Check if it's a .tar.zst file
    tar_files = [f for f in aicity_files if '.tar' in f]
    if tar_files:
        print(f"\n📦 Extracting {tar_files[0]}...")
        print("   This may take 5-10 minutes...")
        !tar -I zstd -xf {aicity_dir}/{tar_files[0]} -C data/ 2>&1 || echo "Extraction failed, trying without zstd..." && tar -xf {aicity_dir}/{tar_files[0]} -C data/
        print("✓ Extraction completed!")
else:
    print("\n❌ No data dataset found!")

In [ ]:
# Check extracted structure
print("📁 Extracted data structure:\n")
!ls -lh data/

In [ ]:
# Find manifest, vitpose, images paths
import glob

print("🔍 Finding data files...\n")

# Find manifest
manifests = glob.glob("data/**/*.jsonl", recursive=True) + glob.glob("data/**/*.parquet", recursive=True)
manifest_file = [f for f in manifests if 'train' in f.lower() and 'hard' in f.lower()][0] if manifests else None
print(f"Manifest: {manifest_file}")

# Find vitpose
vitpose_files = glob.glob("data/**/*vitpose*.json", recursive=True)
vitpose_file = vitpose_files[0] if vitpose_files else None
print(f"VitPose: {vitpose_file}")

# Find images folder
image_folders = glob.glob("data/**/images", recursive=True) + glob.glob("data/**/train_webp", recursive=True)
image_folder = image_folders[0] if image_folders else None
print(f"Images: {image_folder}")

# Find boxes
boxes_files = glob.glob("data/**/boxes*.json*", recursive=True)
boxes_file = boxes_files[0] if boxes_files else None
print(f"Boxes: {boxes_file}")

In [ ]:
# Create config với paths tìm được
import yaml

with open("configs/stage1_30k_kaggle_t4.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Update paths
config['data']['manifest'] = manifest_file
config['data']['image_root'] = image_folder + '/'
config['data']['vitpose_json'] = vitpose_file
config['data']['boxes_json'] = boxes_file
config['model']['checkpoint'] = 'data/checkpoints/best.pth'

# Save
config_path = 'configs/stage1_runtime.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✓ Config saved: {config_path}\n")
print("📋 Paths:")
for k, v in config['data'].items():
    if isinstance(v, str) and (v.startswith('data/') or v.endswith('.pth')):
        print(f"  {k}: {v}")

In [ ]:
# Verify all paths exist
print("🔍 Verifying paths...\n")

paths = {
    'Manifest': manifest_file,
    'VitPose': vitpose_file,
    'Boxes': boxes_file,
    'Images': image_folder,
    'Checkpoint': 'data/checkpoints/best.pth',
}

all_ok = True
for name, path in paths.items():
    if path and Path(path).exists():
        print(f"✓ {name}: {path}")
    else:
        print(f"❌ {name}: NOT FOUND")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Some files missing!")

print("\n✅ All paths verified!")

In [ ]:
# Sanity check: Overfit 1 batch
print("🧪 Sanity check (overfit one batch)...\n")

!python scripts/train.py \
    --config configs/stage1_runtime.yaml \
    --init-from data/checkpoints/best.pth \
    --overfit-one-batch

print("\n✅ Sanity check passed!")

In [ ]:
# FULL TRAINING
print("🚀 Starting full training...\n")

!python scripts/train.py \
    --config configs/stage1_runtime.yaml \
    --init-from data/checkpoints/best.pth \
    --max-hours 11.5

print("\n🎉 Training completed!")

In [ ]:
# Evaluate & save
import torch

best_ckpt = Path("outputs/stage1_30k_t4/best.pth")
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location='cpu')
    report = ckpt.get('report', {})
    
    print("📊 RESULTS:")
    print(f"  mAP: {report.get('mAP', 0)*100:.2f}%")
    print(f"  R@1: {report.get('R@1', 0)*100:.2f}%")
    print(f"  R@5: {report.get('R@5', 0)*100:.2f}%")
    
    # Save to /kaggle/working
    shutil.copy(best_ckpt, "/kaggle/working/stage1_best.pth")
    print("\n✓ Saved to /kaggle/working/stage1_best.pth")
    print("📥 Download from Output tab")
else:
    print("⚠️ No checkpoint found!")